In [8]:
import pandas as pd
from datetime import datetime

fighter_stats = pd.read_csv("ufc-fighters-statistics.csv")
fighter_stats.head()

# Drop column 'nickname'
fighter_stats = fighter_stats.drop(columns=['nickname'])

# Replace missing values with the median in 'height_cm'
fighter_stats = fighter_stats.fillna({'height_cm': fighter_stats['height_cm'].median()})

# Replace missing values with the median in 'weight_in_kg'
fighter_stats = fighter_stats.fillna({'weight_in_kg': fighter_stats['weight_in_kg'].median()})

# Replace missing values with the mean, as it is balanced, in 'weight_in_kg'
fighter_stats = fighter_stats.fillna({'weight_in_kg': fighter_stats['weight_in_kg'].mean()})

# Replace missing values with the mean, as it is balanced, in 'reach_in_cm'
fighter_stats = fighter_stats.fillna({'reach_in_cm': fighter_stats['reach_in_cm'].mean()})

# Change 'date_of_birth' column to 'age'
fighter_stats['date_of_birth'] = pd.to_datetime(fighter_stats['date_of_birth'])

today = pd.Timestamp.today()

def calculate_age(dob):
    if pd.isnull(dob):
        return 0
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
fighter_stats['age'] = fighter_stats['date_of_birth'].apply(calculate_age)

fighter_stats = fighter_stats.drop(columns=['date_of_birth']) # Information changed to numeric 'age' column so can drop 'date_of_birth' column

# Replace missing values with the median in 'age'
fighter_stats = fighter_stats.fillna({'age': fighter_stats['age'].median()})

# Replace missing values with the median in 'significant_strikes_landed_per_minute'
fighter_stats = fighter_stats.fillna({'significant_strikes_landed_per_minute': fighter_stats['significant_strikes_landed_per_minute'].median()})

# Replace missing values with the median in 'significant_striking_accuracy'
fighter_stats = fighter_stats.fillna({'significant_striking_accuracy': fighter_stats['significant_striking_accuracy'].median()})

# Replace missing values with the median in 'significant_strikes_absorbed'
fighter_stats = fighter_stats.fillna({'significant_strikes_absorbed_per_minute': fighter_stats['significant_strikes_absorbed_per_minute'].median()})

# Replace missing values with the median in 'significant_strike_defence'
fighter_stats = fighter_stats.fillna({'significant_strike_defence': fighter_stats['significant_strike_defence'].median()})

# Replace missing values with the median in 'average_takedowns_landed_per_15_minutes'
fighter_stats = fighter_stats.fillna({'average_takedowns_landed_per_15_minutes': fighter_stats['average_takedowns_landed_per_15_minutes'].median()})

# Replace missing values with the median in 'takedown_accuracy'
fighter_stats = fighter_stats.fillna({'takedown_accuracy': fighter_stats['takedown_accuracy'].median()})

# Replace missing values with the median in 'takedown_defense'
fighter_stats = fighter_stats.fillna({'takedown_defense': fighter_stats['takedown_defense'].median()})

# Replace missing values with the median in 'average_submissions_attempted_per_15_minutes'
fighter_stats = fighter_stats.fillna({'average_submissions_attempted_per_15_minutes': fighter_stats['average_submissions_attempted_per_15_minutes'].median()})

# Replace missing values with 'Unknown' in 'stance'
fighter_stats = fighter_stats.fillna({'stance': 'Unknown'})

# One-Hot Encoding to encode the 'stance' column
fighter_stats = pd.get_dummies(fighter_stats, columns=['stance'], prefix='stance', dtype=int)
fighter_stats


,name,wins,losses,draws,height_cm,weight_in_kg,reach_in_cm,significant_strikes_landed_per_minute,significant_striking_accuracy,significant_strikes_absorbed_per_minute,...,takedown_accuracy,takedown_defense,average_submissions_attempted_per_15_minutes,age,stance_Open Stance,stance_Orthodox,stance_Sideways,stance_Southpaw,stance_Switch,stance_Unknown
0,Robert Drysdale,7,0,0,190.50,92.99,181.808874,0.00,0.0,0.00,...,100.0,0.0,21.9,43,0,1,0,0,0,0
1,Daniel McWilliams,15,37,0,185.42,83.91,181.808874,3.36,77.0,0.00,...,0.0,100.0,21.6,0,0,0,0,0,0,1
2,Dan Molina,13,9,0,177.80,97.98,181.808874,0.00,0.0,5.58,...,0.0,0.0,20.9,0,0,0,0,0,0,1
3,Paul Ruiz,7,4,0,167.64,61.23,181.808874,1.40,33.0,1.40,...,0.0,100.0,20.9,0,0,0,0,0,0,1
4,Collin Huckbody,8,2,0,190.50,83.91,193.040000,2.05,60.0,2.73,...,100.0,0.0,20.4,30,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4106,John Campetella,0,1,0,175.26,106.59,181.808874,0.00,0.0,0.00,...,0.0,0.0,0.0,0,0,1,0,0,0,0
4107,Andre Pederneiras,1,1,2,172.72,70.31,181.808874,0.00,0.0,0.00,...,0.0,0.0,0.0,58,0,1,0,0,0,0
4108,Bryson Kamaka,12,20,1,180.34,77.11,181.808874,9.47,60.0,12.63,...,0.0,100.0,0.0,0,0,1,0,0,0,0
4109,Matej Penaz,6,1,0,190.50,83.91,210.820000,1.28,33.0,2.55,...,0.0,0.0,0.0,28,0,0,0,1,0,0


In [9]:
def clean_name(name):
    return name.lower().strip().replace('.', '').replace("'", "")

fighter_stats['name'] = fighter_stats['name'].apply(clean_name)

fight_results = pd.read_csv("ufc_fight_results.csv")

fight_results[['fighter_1', 'fighter_2']] = fight_results['BOUT'].str.split('vs. ', expand=True)
fight_results['fighter_1'] = fight_results['fighter_1'].apply(clean_name)
fight_results['fighter_2'] = fight_results['fighter_2'].apply(clean_name)

# Melt both sides of the fight so each row represents a fighter
fight_df1 = fight_results.rename(columns={'fighter_1': 'name', 'OUTCOME': 'outcome_raw'})
fight_df2 = fight_results.rename(columns={'fighter_2': 'name', 'OUTCOME': 'outcome_raw'})

fight_df1['outcome'] = fight_df1['outcome_raw'].str[0]  # 'W' or 'L'
fight_df2['outcome'] = fight_df2['outcome_raw'].str[-1] # 'L' or 'W'

all_fights = pd.concat([fight_df1, fight_df2])[['EVENT', 'name', 'outcome', 'WEIGHTCLASS', 'METHOD', 'ROUND', 'TIME']]
all_fights = all_fights.sort_values(by='EVENT') # Switch to ordinal encoding when possible

fight_results


,EVENT,BOUT,OUTCOME,WEIGHTCLASS,METHOD,ROUND,TIME,TIME FORMAT,REFEREE,DETAILS,URL,fighter_1,fighter_2
0,UFC 314: Volkanovski vs. Lopes,Alexander Volkanovski vs. Diego Lopes,W/L,UFC Featherweight Title Bout,Decision - Unanimous,5,5:00,5 Rnd (5-5-5-5-5),Marc Goddard,Sal D'amato 46 - 49.Chris Lee 46 - 49.Derek Cl...,http://ufcstats.com/fight-details/e733f148060b...,alexander volkanovski,diego lopes
1,UFC 314: Volkanovski vs. Lopes,Michael Chandler vs. Paddy Pimblett,L/W,Lightweight Bout,KO/TKO,3,3:07,5 Rnd (5-5-5-5-5),Kerry Hatley,Elbows to Head From Mount,http://ufcstats.com/fight-details/d05cb4c4135c...,michael chandler,paddy pimblett
2,UFC 314: Volkanovski vs. Lopes,Yair Rodriguez vs. Patricio Freire,W/L,Featherweight Bout,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Andrew Glenn,Eliseo Rodriguez 27 - 30.Junichiro Kamijo 27 -...,http://ufcstats.com/fight-details/d3be5a4e0ec2...,yair rodriguez,patricio freire
3,UFC 314: Volkanovski vs. Lopes,Bryce Mitchell vs. Jean Silva,L/W,Featherweight Bout,Submission,2,3:52,3 Rnd (5-5-5),Mike Beltran,Guillotine Choke In Clinch,http://ufcstats.com/fight-details/8c540eb4afe8...,bryce mitchell,jean silva
4,UFC 314: Volkanovski vs. Lopes,Nikita Krylov vs. Dominick Reyes,L/W,Light Heavyweight Bout,KO/TKO,1,2:24,3 Rnd (5-5-5),Marc Goddard,Punch to Head At Distance,http://ufcstats.com/fight-details/b2d731415bd3...,nikita krylov,dominick reyes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8076,UFC 2: No Way Out,Orlando Wiet vs. Robert Lucarelli,W/L,Open Weight Bout,KO/TKO,1,2:50,No Time Limit,John McCarthy,toCorner Stoppage,http://ufcstats.com/fight-details/3b020d4914b4...,orlando wiet,robert lucarelli
8077,UFC 2: No Way Out,Frank Hamaker vs. Thaddeus Luster,W/L,Open Weight Bout,Submission,1,4:52,No Time Limit,John McCarthy,Keylock From Half Guard,http://ufcstats.com/fight-details/d917c8c7461b...,frank hamaker,thaddeus luster
8078,UFC 2: No Way Out,Johnny Rhodes vs. David Levicki,W/L,Open Weight Bout,KO/TKO,1,12:13,No Time Limit,John McCarthy,Punches to Head From GuardSubmission to Strikes,http://ufcstats.com/fight-details/ccee020be2e8...,johnny rhodes,david levicki
8079,UFC 2: No Way Out,Patrick Smith vs. Ray Wizard,W/L,Open Weight Bout,Submission,1,0:58,No Time Limit,John McCarthy,Guillotine Choke Standing,http://ufcstats.com/fight-details/4b9ae533ccb3...,patrick smith,ray wizard


In [10]:
# Rolling win rate
all_fights['win'] = all_fights['outcome'] == 'W'
all_fights['win'] = all_fights['win'].astype(int)


all_fights['rolling_win_rate'] = (
    all_fights
    .groupby('name')['win']
    .transform(lambda x: x.rolling(5, min_periods=1).mean().shift())
)

all_fights

,EVENT,name,outcome,WEIGHTCLASS,METHOD,ROUND,TIME,win,rolling_win_rate
7450,Ortiz vs Shamrock 3: The Final Chapter,nate marquardt,W,Middleweight Bout,Submission,2,1:14,1,NaN
7448,Ortiz vs Shamrock 3: The Final Chapter,jason macdonald,W,Middleweight Bout,Submission,1,2:43,1,NaN
7451,Ortiz vs Shamrock 3: The Final Chapter,tony desouza,W,Welterweight Bout,Submission,1,3:59,1,NaN
7454,Ortiz vs Shamrock 3: The Final Chapter,forrest petz,L,Welterweight Bout,Submission,1,4:58,0,NaN
7453,Ortiz vs Shamrock 3: The Final Chapter,john alessio,L,Welterweight Bout,Decision - Unanimous,3,5:00,0,NaN
...,...,...,...,...,...,...,...,...,...
7122,UFC: Silva vs Irvin,hermes franca,L,Lightweight Bout,Decision - Unanimous,3,5:00,0,0.8
7120,UFC: Silva vs Irvin,anderson silva,W,Light Heavyweight Bout,KO/TKO,1,1:01,1,0.6
7130,UFC: Silva vs Irvin,dale hartt,L,Lightweight Bout,Submission,1,3:33,0,0.5
7123,UFC: Silva vs Irvin,jake obrien,L,Heavyweight Bout,KO/TKO,1,2:02,0,0.8


In [11]:
# Drop NA in case some fighters only had 1 match, etc.
latest_rolling_win = (
    all_fights
    .dropna(subset=['rolling_win_rate'])
    .groupby('name', as_index=False)
    .tail(1)  # gets the last fight entry (i.e., most recent rolling stat)
    [['name', 'rolling_win_rate']]
    .drop_duplicates(subset='name')  # ensure no duplicates just in case
)

fighter_stats = fighter_stats.merge(
    latest_rolling_win,
    on='name',
    how='left'
)


fighter_stats

,name,wins,losses,draws,height_cm,weight_in_kg,reach_in_cm,significant_strikes_landed_per_minute,significant_striking_accuracy,significant_strikes_absorbed_per_minute,...,takedown_defense,average_submissions_attempted_per_15_minutes,age,stance_Open Stance,stance_Orthodox,stance_Sideways,stance_Southpaw,stance_Switch,stance_Unknown,rolling_win_rate
0,robert drysdale,7,0,0,190.50,92.99,181.808874,0.00,0.0,0.00,...,0.0,21.9,43,0,1,0,0,0,0,NaN
1,daniel mcwilliams,15,37,0,185.42,83.91,181.808874,3.36,77.0,0.00,...,100.0,21.6,0,0,0,0,0,0,1,NaN
2,dan molina,13,9,0,177.80,97.98,181.808874,0.00,0.0,5.58,...,0.0,20.9,0,0,0,0,0,0,1,NaN
3,paul ruiz,7,4,0,167.64,61.23,181.808874,1.40,33.0,1.40,...,100.0,20.9,0,0,0,0,0,0,1,NaN
4,collin huckbody,8,2,0,190.50,83.91,193.040000,2.05,60.0,2.73,...,0.0,20.4,30,0,1,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4106,john campetella,0,1,0,175.26,106.59,181.808874,0.00,0.0,0.00,...,0.0,0.0,0,0,1,0,0,0,0,NaN
4107,andre pederneiras,1,1,2,172.72,70.31,181.808874,0.00,0.0,0.00,...,0.0,0.0,58,0,1,0,0,0,0,NaN
4108,bryson kamaka,12,20,1,180.34,77.11,181.808874,9.47,60.0,12.63,...,100.0,0.0,0,0,1,0,0,0,0,NaN
4109,matej penaz,6,1,0,190.50,83.91,210.820000,1.28,33.0,2.55,...,0.0,0.0,28,0,0,0,1,0,0,NaN


In [12]:
fighter_stats['rolling_win_rate'] = fighter_stats['rolling_win_rate'].fillna(fighter_stats['rolling_win_rate'].mean())

fighter_stats[fighter_stats['name'] == 'diego lopes']
fight_data = None

In [13]:
fight_data = fight_results.merge(
    fighter_stats, left_on="fighter_1", right_on='name', how='left'
).add_prefix('1_')

fight_data = fight_data.merge(
    fighter_stats, left_on='1_fighter_2', right_on='name', how='left'
).add_prefix('2_')

fight_data = fight_data.drop(columns=['2_1_BOUT', '2_1_WEIGHTCLASS', '2_1_METHOD', '2_1_BOUT', '2_1_ROUND', '2_1_TIME', '2_1_TIME FORMAT', '2_1_REFEREE', '2_1_DETAILS', '2_1_URL', '2_1_fighter_1', '2_1_fighter_2'])

In [14]:
fight_data['outcome'] = (fight_data['outcome'] == 'W').astype(int)

fight_data = fight_data.drop(columns=['2_1_OUTCOME'])
fight_data = fight_data.fillna(fight_data.mean(numeric_only=True))

fight_data.to_csv('fight_data_final.csv')

KeyError: 'outcome'

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

fight_data = pd.read_csv("fight_data_final.csv")
X = fight_data.drop(columns=['outcome', '2_1_name', '2_name', '2_1_EVENT'])
y = fight_data['outcome']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


print("\nROC AUC Score:", roc_auc_score(y_test, y_proba))

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 0

In [1]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=7)  
knn.fit(X_train_scaled, y_train)

y_knn_pred = knn.predict(X_test_scaled)
y_knn_proba = knn.predict_proba(X_test_scaled)[:, 1]

print("K Nearest Confusion Matrix:\n", confusion_matrix(y_test, y_knn_pred))
print("\nK Nearest Classification Report:\n", classification_report(y_test, y_knn_pred))
print("\nROC AUC Score:", roc_auc_score(y_test, y_knn_proba))

NameError: name 'X_train_scaled' is not defined

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=1000, random_state=42)
rf.fit(X_train, y_train)

y_rf_pred = rf.predict(X_test)
y_rf_proba = rf.predict_proba(X_test)[:, 1]  # for ROC AUC

print("Random Forest - Confusion Matrix:")
print(confusion_matrix(y_test, y_rf_pred))

print("\nRandom Forest - Classification Report:")
print(classification_report(y_test, y_rf_pred))

print("\nRandom Forest - ROC AUC Score:", roc_auc_score(y_test, y_rf_proba))

Random Forest - Confusion Matrix:
[[335 257]
 [140 892]]

Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.57      0.63       592
           1       0.78      0.86      0.82      1032

    accuracy                           0.76      1624
   macro avg       0.74      0.72      0.72      1624
weighted avg       0.75      0.76      0.75      1624


Random Forest - ROC AUC Score: 0.8162843075633773


In [13]:
fight_data['outcome'].value_counts()

outcome
1    5159
0    2959
Name: count, dtype: int64

In [ ]:
import numpy as np
from random import shuffle
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(penalty='l1', solver='saga', max_iter=1000))
])


# potential hyperparameters and other parameters. 
params = {
  'lr__C': [0.001,0.01,0.1,1,10,100],
}


grid = GridSearchCV(
    pipeline,
    param_grid=params,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X, y)


print("Best C:", grid.best_params_['lr__C'])
print("Best CV AUC:", grid.best_score_)

best_model = grid.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)
print("Test AUC:     ", roc_auc_score(y_test, y_pred_proba))
print("Test Accuracy:", accuracy_score(y_test,  y_pred))
print("\nClassification Report:\n",
      classification_report(y_test, y_pred))

Best C: 0.01
Best CV AUC: 0.7296066098534316
Test AUC:      0.811124260161324
Test Accuracy: 0.7561576354679803

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.55      0.62       592
           1       0.77      0.88      0.82      1032

    accuracy                           0.76      1624
   macro avg       0.74      0.71      0.72      1624
weighted avg       0.75      0.76      0.75      1624



In [28]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(penalty='l2', solver='saga', max_iter=1000))
])


# potential hyperparameters and other parameters. 
params = {
  'lr__C': [0.001,0.01,0.1,1,10,100],
}


grid = GridSearchCV(
    pipeline,
    param_grid=params,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X, y)


print("Best C:", grid.best_params_['lr__C'])
print("Best CV AUC:", grid.best_score_)

best_model = grid.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)
print("Test AUC:     ", roc_auc_score(y_test, y_pred_proba))
print("Test Accuracy:", accuracy_score(y_test,  y_pred))
print("\nClassification Report:\n",
      classification_report(y_test, y_pred))

Best C: 0.01
Best CV AUC: 0.7292312294618941
Test AUC:      0.8130065930756338
Test Accuracy: 0.75

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.57      0.63       592
           1       0.78      0.85      0.81      1032

    accuracy                           0.75      1624
   macro avg       0.73      0.71      0.72      1624
weighted avg       0.74      0.75      0.74      1624



In [27]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000))
])


# potential hyperparameters and other parameters. 
params = {
  'lr__C': [0.001,0.01,0.1,1,10,100],
}


grid = GridSearchCV(
    pipeline,
    param_grid=params,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X, y)


print("Best C:", grid.best_params_['lr__C'])
print("Best CV AUC:", grid.best_score_)

best_model = grid.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)
print("Test AUC:     ", roc_auc_score(y_test, y_pred_proba))
print("Test Accuracy:", accuracy_score(y_test,  y_pred))
print("\nClassification Report:\n",
      classification_report(y_test, y_pred))

Best C: 0.01
Best CV AUC: 0.7292312294618941
Test AUC:      0.813013140320553
Test Accuracy: 0.75

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.57      0.63       592
           1       0.78      0.85      0.81      1032

    accuracy                           0.75      1624
   macro avg       0.73      0.71      0.72      1624
weighted avg       0.74      0.75      0.74      1624



In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000))
])


# potential hyperparameters and other parameters. 
params = {
  'lr__penalty':    ['l2','l1','elasticnet'],
  'lr__C':          [0.001,0.01,0.1,1,10,100],
  'lr__solver':     ['lbfgs','saga','newton-cg'],
  'lr__l1_ratio':   [0.0,0.25,0.5,0.75,1.0],       # only used for elasticnet
  'lr__class_weight': ['balanced', {0:1,1:2}, {0:1,1:3}]
}


grid = GridSearchCV(
    pipeline,
    param_grid=params,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X, y)


print("Best C:", grid.best_params_['lr__C'])
print("Best CV AUC:", grid.best_score_)

best_model = grid.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)
print("Test AUC:     ", roc_auc_score(y_test, y_pred_proba))
print("Test Accuracy:", accuracy_score(y_test,  y_pred))
print("\nClassification Report:\n",
      classification_report(y_test, y_pred))

c:\Users\82cha\miniconda3\envs\myenv\lib\site-packages\sklearn\model_selection\_validation.py:425: FitFailedWarning: 
3600 fits failed out of a total of 8100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
900 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\82cha\miniconda3\envs\myenv\lib\site-packages\sklearn\model_selection\_validation.py", line 732, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\82cha\miniconda3\envs\myenv\lib\site-packages\sklearn\base.py", line 1151, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\82cha\miniconda3\envs\myenv\lib\site-packages\sklearn\pipeline.py", line 420, in fit
    self._final_estimator.

Best C: 10
Best CV AUC: 0.7133381521323185
Test AUC:      0.8122765752671275
Test Accuracy: 0.7426108374384236

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.37      0.51       592
           1       0.73      0.95      0.82      1032

    accuracy                           0.74      1624
   macro avg       0.78      0.66      0.67      1624
weighted avg       0.76      0.74      0.71      1624



In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score

# 1) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2) Base estimator
rf = RandomForestClassifier(
    class_weight='balanced_subsample',  # balance per bootstrap sample
    random_state=42,
    n_jobs=-1
)

# 3) Parameter distributions
param_dist = {
    'n_estimators':      [200, 500, 1000, 1500],
    'max_depth':         [None, 10, 20, 30, 50],
    'max_features':      ['sqrt', 'log2', 0.2, 0.5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'bootstrap':         [True, False]
}

# 4) Randomized search
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,               # number of random draws
    scoring='roc_auc',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV AUC:", search.best_score_)

# 5) Evaluate on test set
best_rf = search.best_estimator_
y_proba = best_rf.predict_proba(X_test)[:,1]
print("Test AUC:", roc_auc_score(y_test, y_proba))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'n_estimators': 1500, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 10, 'bootstrap': True}
Best CV AUC: 0.8123926596782635
Test AUC: 0.8189318497276347


In [ ]:
y_pred = best_rf.predict(X_test)
print("\nBest RF Classification Report:\n",
      classification_report(y_test, y_pred))
fight_data


Best RF Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.73      0.67       592
           1       0.83      0.75      0.79      1032

    accuracy                           0.74      1624
   macro avg       0.73      0.74      0.73      1624
weighted avg       0.75      0.74      0.74      1624

